<a href="https://colab.research.google.com/github/guneetnagia/LearnML/blob/main/SimpleTransformerdch27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "transformers<5" tf-keras

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [ ]:
import tensorflow as tf
import tf_keras as keras
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordLevelTrainer
from transformers import PreTrainedTokenizerFast,GPT2Config,TFAutoModelForCausalLM,DataCollatorForLanguageModeling,TFGPT2LMHeadModel

In [ ]:
import numpy as np

In [ ]:
from datasets import Dataset
np.set_printoptions(precision=3,suppress=True)
tf.random.set_seed(42)

In [ ]:
sentences=[
    "a cat sat on the mat.",
    "the dog sat on the floor.",
    "a cat chased the dog .",
    "the mat is on the floor.",
    "the dog isun beleiv able"
]

In [ ]:
backend_tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
backend_tokenizer.pre_tokenizer = Whitespace()
trainer = WordLevelTrainer(special_tokens=["[UNK]","[PAD]"])
backend_tokenizer.train_from_iterator(sentences,trainer=trainer)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tokenizer,pad_token="[PAD]",unk_token="[UNK]")

In [ ]:
text = "a cat sat on the mat ."
tokens = tokenizer.tokenize(text)
tokens

['a', 'cat', 'sat', 'on', 'the', 'mat', '.']

In [ ]:
print("vocaublary")
for word,idx in sorted(tokenizer.get_vocab().items(),key=lambda x:x[1]):
    print(f"{idx:2} : {word}")

vocaublary
 0 : [UNK]
 1 : [PAD]
 2 : the
 3 : .
 4 : dog
 5 : on
 6 : a
 7 : cat
 8 : floor
 9 : mat
10 : sat
11 : able
12 : beleiv
13 : chased
14 : is
15 : isun


In [ ]:
text = "a cat sat on the"
encoded = tokenizer(text,return_tensors="tf")
print("text")
print(text)
print("tokens")
print(tokenizer.tokenize(text))
print("token ids")
print(encoded["input_ids"].numpy())


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


text
a cat sat on the
tokens
['a', 'cat', 'sat', 'on', 'the']
token ids
[[ 6  7 10  5  2]]


In [ ]:
ids = encoded["input_ids"][0]
one_hot = tf.one_hot(ids,depth=tokenizer.vocab_size)
print(one_hot.numpy().astype(int))

[[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]]


In [ ]:
config = GPT2Config(
    vocab_size = tokenizer.vocab_size,
    n_positions=32,
    n_embd=32,
    n_layer=2,
    n_head=4,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=False,
    output_hidden_states=True,
    output_attentions=True
)

In [ ]:
model = TFGPT2LMHeadModel(config)

The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [ ]:
dummy = tokenizer("a cat",return_tensors = "tf")
_ = model(dummy)

In [ ]:
model.summary()


Model: "tfgpt2lm_head_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFGPT2MainLay  multiple                  27008     
 er)                                                             
                                                                 
Total params: 27008 (105.50 KB)
Trainable params: 27008 (105.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
cat_id = tokenizer.convert_tokens_to_ids("cat")
cat_before = model.transformer.wte(tf.constant([[cat_id]]))
print("cat embeddings beofre tain")
print(cat_before.numpy())

cat embeddings beofre tain
[[[ 0.017  0.014 -0.024  0.022 -0.02  -0.017  0.003  0.008 -0.015  0.02
   -0.018 -0.021  0.001  0.002 -0.001  0.017 -0.011  0.004  0.015 -0.028
    0.005 -0.022  0.004 -0.012 -0.029  0.028  0.035  0.004 -0.033  0.009
   -0.004  0.01 ]]]


In [ ]:
dataset = Dataset.from_dict({"text":sentences})

In [ ]:
def tokenize_data(batch):
  return tokenizer(batch["text"],truncation=True,max_length=16)
tokenized_dataset = dataset.map(tokenize_data,batched=True,remove_columns=["text"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False,return_tensors="tf")

In [ ]:
train_dataset = model.prepare_tf_dataset(tokenized_dataset,shuffle=True,batch_size=5,collate_fn=data_collator)

In [ ]:
optimzer = tf.keras.optimizers.Adam(learning_rate=0.003)

In [ ]:
model.compile(optimizer=optimzer)

In [ ]:
history = model.fit(train_dataset,epochs=400,verbose=1)
print("final loss",history.history["loss"][-1])

Epoch 1/400
1/1 [==============================] - 23s 23s/step - loss: 2.8130
Epoch 2/400
1/1 [==============================] - 0s 57ms/step - loss: 2.6160
Epoch 3/400
1/1 [==============================] - 0s 102ms/step - loss: 2.4854
Epoch 4/400
1/1 [==============================] - 0s 93ms/step - loss: 2.4012
Epoch 5/400
1/1 [==============================] - 0s 63ms/step - loss: 2.3145
Epoch 6/400
1/1 [==============================] - 0s 64ms/step - loss: 2.2375
Epoch 7/400
1/1 [==============================] - 0s 58ms/step - loss: 2.1651
Epoch 8/400
1/1 [==============================] - 0s 64ms/step - loss: 2.0959
Epoch 9/400
1/1 [==============================] - 0s 62ms/step - loss: 2.0257
Epoch 10/400
1/1 [==============================] - 0s 40ms/step - loss: 1.9556
Epoch 11/400
1/1 [==============================] - 0s 29ms/step - loss: 1.8858
Epoch 12/400
1/1 [==============================] - 0s 35ms/step - loss: 1.8173
Epoch 13/400
1/1 [==============================

In [ ]:
text = "a cat sat on the"
inputs = tokenizer(text,return_tensors="tf")
outputs = model(inputs)

In [ ]:
next_token_logits = outputs.logits[0,-1]
predicted_id = tf.argmax(next_token_logits).numpy()
predicted_word = tokenizer.convert_ids_to_tokens(int(predicted_id))
print("Input :",text)
print(f"predicted word: {predicted_word}")

Input : a cat sat on the
predicted word: mat


In [ ]:
probabilities = tf.nn.softmax(next_token_logits)
confidence = probabilities[predicted_id].numpy()
print("Input:",text)
print("Prediction",predicted_word)
print("probabilities:",round(confidence*100,2),"%")

Input: a cat sat on the
Prediction mat
probabilities: 99.68 %
